# Choosing the active space

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

Which electrons and orbitals must the calculation treat explicitly?

</div>

</div>

## Learning objectives

After completing this chapter, you will be able to:

- Explain why strongly correlated systems require more than one determinant.
- Distinguish inactive, active, and virtual orbitals.
- Select a valence active space with QDK/Chemistry.
- Explain how natural orbitals and reduced density matrices support active-space selection.
- Explain how orbital entropies can refine an active-space choice.
- Evaluate the tradeoff between active-space accuracy and problem size.

## Before you begin

This course requires a Python environment with the `qdk-chemistry[jupyter]` package.

`qdk-chemistry` ships compiled binaries for Linux, macOS on Apple silicon, and Windows on x86-64. This course also needs PySCF, which has no Windows build, so run it inside WSL on Windows. Run the cell below to check the current environment.

In [ ]:
# If packages are missing, first select a dedicated Python environment/kernel,
# then uncomment the next line and run this cell again.
# %pip install -r ../requirements.txt

from _unit import check_env

check_env()

## Setting up

The cell below imports the QDK/Chemistry pieces this chapter uses and quiets the solver logs.

In [ ]:
from dataclasses import dataclass
from typing import TYPE_CHECKING, cast

from qdk_chemistry.algorithms import create
from qdk_chemistry.data import Orbitals, Structure, Wavefunction
from qdk_chemistry.data.symmetry import SymmetryLabel, axes
from qdk_chemistry.utils import Logger, compute_valence_space_parameters
from tutorial_orbital_coordinates import (
    NaturalOrbitalCoordinateMinimizationResult,
    coordinate_minimize_natural_orbital_coefficient_norm,
)

if TYPE_CHECKING:
    from matplotlib.figure import Figure

from tutorial_choose_active_space import create_stretched_n2_structure

structure = create_stretched_n2_structure()
charge = 0
spin_multiplicity = 1
basis_set = "cc-pvdz"

scf_solver = create("scf_solver", "qdk")
hartree_fock_energy, hartree_fock_wavefunction = scf_solver.run(
    structure,
    charge=charge,
    spin_multiplicity=spin_multiplicity,
    basis_or_guess=basis_set,
)

## The limits of one determinant

As the previous chapter explains, the [Hartree–Fock method ↗](https://en.wikipedia.org/wiki/Hartree%E2%80%93Fock_method) restricts the [wavefunction ↗](https://en.wikipedia.org/wiki/Wave_function) to one optimized [Slater determinant ↗](https://en.wikipedia.org/wiki/Slater_determinant).
This determinant represents one [electron configuration ↗](https://en.wikipedia.org/wiki/Electron_configuration), a pattern of occupied spin orbitals introduced in *Orbitals and determinants*.
This description is often a useful starting point near an equilibrium geometry, where one configuration dominates the ground-state wavefunction, as discussed for N<sub>2</sub> in *Specify the molecular system* chapter.
Stretching a chemical bond can make several configurations similar in energy because electrons can no longer be assigned adequately to one fixed pattern of occupied and unoccupied [molecular orbitals ↗](https://en.wikipedia.org/wiki/Molecular_orbital_theory).
The need to combine these multiple important configurations is called [static correlation ↗](https://en.wikipedia.org/wiki/Electronic_correlation).
The stretched N<sub>2</sub> geometry has been selected to demonstrate this regime.
The correlated calculations below evaluate static correlation by constructing a multi-determinant wavefunction and measuring orbital-occupation entropies.

A [configuration interaction ↗](https://en.wikipedia.org/wiki/Configuration_interaction) (CI) calculation addresses this limitation by calculating a wavefunction expanded in multiple Slater determinants:

$$
\vert \Psi \rangle = \sum_I c_I \vert \Phi_I \rangle,
$$

where $\vert \Phi_I \rangle$ is determinant $I$ and $c_I$ is its coefficient.
Every determinant $\Phi_I$, including $\Phi_{\mathrm{HF}}$, is constructed from one allowed choice of occupied spin orbitals.
Different choices represent different electron configurations.
Full configuration interaction (FCI) includes every determinant consistent with a chosen finite orbital basis and fixed $(n_\alpha,n_\beta)$ sector.
It therefore gives the exact eigenvalues of the finite-basis Hamiltonian in that sector, up to numerical solver tolerance.
For this 14-electron calculation in 28 `cc-pvdz` spatial orbitals, the $(n_\alpha,n_\beta)=(7,7)$ sector contains

$$
\binom{28}{7}\binom{28}{7}
= 1{,}401{,}950{,}721{,}600
\approx 1.4\times 10^{12}
$$

determinants.
At a fixed electron-to-orbital ratio, this count grows exponentially with the number of orbitals, making full-basis FCI impractical for large systems.
An [active-space model ↗](https://en.wikipedia.org/wiki/Complete_active_space) controls that cost by allowing occupations to vary only among selected orbitals, trading some model accuracy for a smaller determinant space.
If that model contains $n_o$ active spatial orbitals, $n_\alpha$ active $\alpha$ electrons, and $n_\beta$ active $\beta$ electrons, the choices of occupied $\alpha$ and $\beta$ spin orbitals give

$$
N_{\mathrm{det}} = \binom{n_o}{n_\alpha}\binom{n_o}{n_\beta}.
$$

## The active space

<div style="text-align:center;">

<svg width="802pt" height="171pt" viewBox="0.00 0.00 802.00 171.00" xmlns="http://www.w3.org/2000/svg" xmlns:xlink="http://www.w3.org/1999/xlink" class="qdk-chemistry-orbital-partition" role="img" data-asset="tutorial_qpe_orbital_partition.svg" aria-label="The spatial molecular orbitals are partitioned into inactive orbitals that remain doubly occupied and contribute to the core energy, active orbitals whose occupations vary among determinants and whose correlation is treated explicitly, and virtual orbitals that remain empty and are excluded from the correlated calculation." style="max-width:100%;height:auto"><style>.qdk-chemistry-orbital-partition { --qdk-fg: var(--vscode-editor-foreground, var(--jp-content-font-color1, currentColor)); } .qdk-chemistry-orbital-partition path[fill="#ffffff"], .qdk-chemistry-orbital-partition polygon[fill="#ffffff"] { fill: transparent; } .qdk-chemistry-orbital-partition path[fill="lightgrey"], .qdk-chemistry-orbital-partition polygon[fill="lightgrey"] { fill: none; } .qdk-chemistry-orbital-partition path[fill="#e8eaf6"], .qdk-chemistry-orbital-partition polygon[fill="#e8eaf6"] { fill: color-mix(in srgb, #3949ab 16%, transparent); } .qdk-chemistry-orbital-partition path[fill="#e0f2f1"], .qdk-chemistry-orbital-partition polygon[fill="#e0f2f1"] { fill: color-mix(in srgb, #00796b 16%, transparent); } .qdk-chemistry-orbital-partition path[fill="#e3f2fd"], .qdk-chemistry-orbital-partition polygon[fill="#e3f2fd"] { fill: color-mix(in srgb, #1976d2 16%, transparent); } .qdk-chemistry-orbital-partition path[fill="#f3e5f5"], .qdk-chemistry-orbital-partition polygon[fill="#f3e5f5"] { fill: color-mix(in srgb, #7b1fa2 16%, transparent); } .qdk-chemistry-orbital-partition text[fill="#004d40"] { fill: var(--qdk-fg); } .qdk-chemistry-orbital-partition [stroke="#004d40"] { stroke: color-mix(in srgb, #004d40 65%, var(--qdk-fg)); } .qdk-chemistry-orbital-partition text[fill="#00796b"] { fill: var(--qdk-fg); } .qdk-chemistry-orbital-partition [stroke="#00796b"] { stroke: color-mix(in srgb, #00796b 65%, var(--qdk-fg)); } .qdk-chemistry-orbital-partition text[fill="#0d47a1"] { fill: var(--qdk-fg); } .qdk-chemistry-orbital-partition [stroke="#0d47a1"] { stroke: color-mix(in srgb, #0d47a1 65%, var(--qdk-fg)); } .qdk-chemistry-orbital-partition text[fill="#1976d2"] { fill: var(--qdk-fg); } .qdk-chemistry-orbital-partition [stroke="#1976d2"] { stroke: color-mix(in srgb, #1976d2 65%, var(--qdk-fg)); } .qdk-chemistry-orbital-partition text[fill="#26a69a"] { fill: var(--qdk-fg); } .qdk-chemistry-orbital-partition [stroke="#26a69a"] { stroke: color-mix(in srgb, #26a69a 65%, var(--qdk-fg)); } .qdk-chemistry-orbital-partition text[fill="#283593"] { fill: var(--qdk-fg); } .qdk-chemistry-orbital-partition [stroke="#283593"] { stroke: color-mix(in srgb, #283593 65%, var(--qdk-fg)); } .qdk-chemistry-orbital-partition text[fill="#3949ab"] { fill: var(--qdk-fg); } .qdk-chemistry-orbital-partition [stroke="#3949ab"] { stroke: color-mix(in srgb, #3949ab 65%, var(--qdk-fg)); } .qdk-chemistry-orbital-partition text[fill="#42a5f5"] { fill: var(--qdk-fg); } .qdk-chemistry-orbital-partition [stroke="#42a5f5"] { stroke: color-mix(in srgb, #42a5f5 65%, var(--qdk-fg)); } .qdk-chemistry-orbital-partition text[fill="#4a148c"] { fill: var(--qdk-fg); } .qdk-chemistry-orbital-partition [stroke="#4a148c"] { stroke: color-mix(in srgb, #4a148c 65%, var(--qdk-fg)); } .qdk-chemistry-orbital-partition text[fill="#5c6bc0"] { fill: var(--qdk-fg); } .qdk-chemistry-orbital-partition [stroke="#5c6bc0"] { stroke: color-mix(in srgb, #5c6bc0 65%, var(--qdk-fg)); } .qdk-chemistry-orbital-partition text[fill="#7b1fa2"] { fill: var(--qdk-fg); } .qdk-chemistry-orbital-partition [stroke="#7b1fa2"] { stroke: color-mix(in srgb, #7b1fa2 65%, var(--qdk-fg)); } .qdk-chemistry-orbital-partition text[fill="#ab47bc"] { fill: var(--qdk-fg); } .qdk-chemistry-orbital-partition [stroke="#ab47bc"] { stroke: color-mix(in srgb, #ab47bc 65%, var(--qdk-fg)); }</style> <g id="graph0" class="graph" transform="scale(1 1) rotate(0) translate(14.4 156.4)"> <title>TutorialQpeOrbitalPartition</title> <g id="clust1" class="cluster"> <title>cluster_orbitals</title> <path fill="#e8eaf6" stroke="#5c6bc0" stroke-width="2.5" d="M20,-8C20,-8 753.6,-8 753.6,-8 759.6,-8 765.6,-14 765.6,-20 765.6,-20 765.6,-122 765.6,-122 765.6,-128 759.6,-134 753.6,-134 753.6,-134 20,-134 20,-134 14,-134 8,-128 8,-122 8,-122 8,-20 8,-20 8,-14 14,-8 20,-8"/> <text xml:space="preserve" text-anchor="middle" x="386.8" y="-116.7" font-family="Arial Bold" font-size="14.00" fill="#3949ab">Spatial molecular orbitals</text> </g> <!-- Inactive --> <g id="node1" class="node"> <title>Inactive</title> <path fill="#1976d2" stroke="#0d47a1" stroke-width="2.5" d="M227.2,-102.2C227.2,-102.2 28,-102.2 28,-102.2 22,-102.2 16,-96.2 16,-90.2 16,-90.2 16,-27.8 16,-27.8 16,-21.8 22,-15.8 28,-15.8 28,-15.8 227.2,-15.8 227.2,-15.8 233.2,-15.8 239.2,-21.8 239.2,-27.8 239.2,-27.8 239.2,-90.2 239.2,-90.2 239.2,-96.2 233.2,-102.2 227.2,-102.2"/> <text xml:space="preserve" text-anchor="start" x="76.23" y="-71.7" font-family="Arial" font-weight="bold" font-size="14.00" fill="#ffffff">Inactive orbitals</text> <text xml:space="preserve" text-anchor="start" x="37.6" y="-46.55" font-family="Arial" font-size="11.00" fill="#e3f2fd">Doubly occupied in every determinant</text> <text xml:space="preserve" text-anchor="start" x="57.48" y="-35.55" font-family="Arial" font-size="11.00" fill="#e3f2fd">Contribute to the core energy</text> </g> <!-- Active --> <g id="node2" class="node"> <title>Active</title> <path fill="#00796b" stroke="#004d40" stroke-width="2.5" d="M486.4,-102.2C486.4,-102.2 287.2,-102.2 287.2,-102.2 281.2,-102.2 275.2,-96.2 275.2,-90.2 275.2,-90.2 275.2,-27.8 275.2,-27.8 275.2,-21.8 281.2,-15.8 287.2,-15.8 287.2,-15.8 486.4,-15.8 486.4,-15.8 492.4,-15.8 498.4,-21.8 498.4,-27.8 498.4,-27.8 498.4,-90.2 498.4,-90.2 498.4,-96.2 492.4,-102.2 486.4,-102.2"/> <text xml:space="preserve" text-anchor="start" x="340.3" y="-71.7" font-family="Arial" font-weight="bold" font-size="14.00" fill="#ffffff">Active orbitals</text> <text xml:space="preserve" text-anchor="start" x="293.43" y="-46.55" font-family="Arial" font-size="11.00" fill="#e0f2f1">Occupation varies among determinants</text> <text xml:space="preserve" text-anchor="start" x="315.18" y="-35.55" font-family="Arial" font-size="11.00" fill="#e0f2f1">Correlation is treated explicitly</text> </g> <!-- Inactive&#45;&gt;Active --> <!-- Virtual --> <g id="node3" class="node"> <title>Virtual</title> <path fill="#7b1fa2" stroke="#4a148c" stroke-width="2.5" d="M745.6,-102.2C745.6,-102.2 546.4,-102.2 546.4,-102.2 540.4,-102.2 534.4,-96.2 534.4,-90.2 534.4,-90.2 534.4,-27.8 534.4,-27.8 534.4,-21.8 540.4,-15.8 546.4,-15.8 546.4,-15.8 745.6,-15.8 745.6,-15.8 751.6,-15.8 757.6,-21.8 757.6,-27.8 757.6,-27.8 757.6,-90.2 757.6,-90.2 757.6,-96.2 751.6,-102.2 745.6,-102.2"/> <text xml:space="preserve" text-anchor="start" x="598.75" y="-71.7" font-family="Arial" font-weight="bold" font-size="14.00" fill="#ffffff">Virtual orbitals</text> <text xml:space="preserve" text-anchor="start" x="580.38" y="-46.55" font-family="Arial" font-size="11.00" fill="#f3e5f5">Empty in every determinant</text> <text xml:space="preserve" text-anchor="start" x="550.38" y="-35.55" font-family="Arial" font-size="11.00" fill="#f3e5f5">Excluded from the correlated calculation</text> </g> <!-- Active&#45;&gt;Virtual --> </g> </svg>

*An active-space calculation varies occupations only among the active orbitals; inactive and virtual occupations remain fixed across determinants.*

</div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why can an inactive orbital still contribute to the molecular energy?</summary>

<div style="padding:0.1em 1em;">

An inactive spatial orbital remains doubly occupied in every determinant.
Its electrons and their interactions contribute to the core part of the active-space Hamiltonian even though the calculation does not vary their occupations.

</div>

</details></div>

A complete active space containing $n_e$ active electrons in $n_o$ active spatial orbitals is written CAS $(n_e,n_o)$.
Complete active space configuration interaction (CASCI) forms every determinant consistent with those active electron and orbital counts while keeping the molecular orbitals fixed.
Unlike complete active space self-consistent field (CASSCF), CASCI does not reoptimize the orbitals.

A useful first choice is a generous valence space containing orbitals on both sides of the occupied–virtual boundary.
The `compute_valence_space_parameters()` function determines the numbers of valence electrons and valence spatial orbitals from the Hartree–Fock wavefunction and molecular charge.
The qdk_valence selector uses those numbers to construct an initial active space from orbitals near the HOMO–LUMO gap.
The highest occupied molecular orbital (HOMO) and lowest unoccupied molecular orbital (LUMO) define the boundary between occupied and virtual orbitals in the Hartree–Fock reference.
Orbitals near this boundary are the most accessible when low-energy configurations redistribute electrons, so a valence window around the gap is a useful generous starting space for the correlated calculation:

In [ ]:
num_valence_electrons, num_valence_orbitals = compute_valence_space_parameters(
    hartree_fock_wavefunction, charge
)
valence_selector = create(
    "active_space_selector",
    "qdk_valence",
    num_active_electrons=num_valence_electrons,
    num_active_orbitals=num_valence_orbitals,
)
valence_wavefunction = valence_selector.run(hartree_fock_wavefunction)

# Restricted alpha and beta channels contain the same spatial-orbital indices,
# so read one channel to count each spatial orbital once.
alpha_channel = SymmetryLabel([axes.alpha()])
valence_indices = list(
    valence_wavefunction.get_orbitals().active_indices().indices(alpha_channel)
)
num_valence_alpha, num_valence_beta = (
    valence_wavefunction.get_active_num_electrons()
)

For this restricted calculation, matching $\alpha$ and $\beta$ channels describe the same spatial orbitals, so the code reads one channel and counts each spatial orbital once.
The script reports the resulting active electron and orbital counts and the zero-based indices of the active orbitals.
Use these values with the total number of `cc-pvdz` molecular orbitals from the chapter *Describing the molecule* to determine the initial partitioning of inactive, active, and virtual orbitals.

## A correlated active-space wavefunction

The active-space selector labels orbitals but does not determine how strongly each orbital participates in correlation.
That evidence must come from a correlated wavefunction.
The script constructs the molecular Hamiltonian in the initial valence space and solves it with the *MACIS CASCI implementation*:

In [ ]:
hamiltonian_constructor = create("hamiltonian_constructor")
casci_solver = create(
    "multi_configuration_calculator",
    "macis_cas",
    # Tight convergence keeps the RDM-derived natural subspaces stable
    # across numerical backends before their orbital gauge is selected.
    ci_residual_tolerance=1e-10,
    # autoCAS entropies require both one- and two-particle RDMs.
    calculate_one_rdm=True,
    calculate_two_rdm=True,
)

valence_hamiltonian = hamiltonian_constructor.run(
    valence_wavefunction.get_orbitals()
)
valence_energy, valence_casci_wavefunction = casci_solver.run(
    valence_hamiltonian,
    num_valence_alpha,
    num_valence_beta,
)
num_valence_determinants = len(valence_casci_wavefunction.get_coefficients())

Using the determinant-count formula above, the initial valence space in this example is small enough to include every determinant rather than approximating the wavefunction with a selected subset.
Larger active-space studies often use selected CI (SCI) to obtain approximate active-space diagnostics at lower cost, but that additional approximation is unnecessary here.

The `calculate_one_rdm` and `calculate_two_rdm` settings request the one- and two-particle reduced density matrices (RDMs).
An RDM summarizes the parts of the many-electron wavefunction needed to describe one- or two-particle properties.
The spin-resolved one-particle RDM tracks $\alpha$ and $\beta$ occupations separately, and its diagonal gives the expected occupation of each spin orbital.
A corresponding diagonal element of the two-particle RDM gives the joint occupation of a pair of spin orbitals.
Each determinant assigns every spatial orbital one of four *local occupation states*: empty, occupied by one $\alpha$ electron, occupied by one $\beta$ electron, or doubly occupied.
Here, a local occupation state describes only one orbital, not the complete electronic state of the molecule.
For these occupation quantities, determinant $I$ contributes with weight $\lvert c_I\rvert^2$, the squared magnitude of its coefficient in the correlated wavefunction.
Together, the RDM elements collect these contributions into the probabilities of the four local occupation states.
If the important determinants give an orbital the same occupation, one probability dominates; if they assign different occupations, the probabilities spread among several local states.

## Natural-orbital transformation

Different sets of molecular orbitals can describe the same active space.
Changing to natural orbitals changes the individual orbital shapes but not the exact CASCI energy for that active space.
Orbital-resolved quantities, however, can change because they describe the chosen orbital representation.

Natural orbitals diagonalize the one-particle RDM.
Their eigenvalues are natural-orbital occupation numbers between zero and two for spatial orbitals.
In this basis, the off-diagonal elements vanish, so each occupation number is associated directly with one natural orbital rather than being mixed among several orbitals.
Occupations near two identify nearly doubly occupied orbitals, occupations near zero identify nearly empty orbitals, and fractional occupations can reveal orbitals that require multiple electron configurations.
Natural orbitals provide a useful convention in which the one-particle occupations are directly associated with individual orbitals before applying the orbital-resolved entropy criterion.

The supported qdk_natural_orbitals transformation uses the one-particle RDM from the initial CASCI wavefunction.
It rotates the active orbitals into the natural-orbital representation described above.
The script then rebuilds and resolves the initial valence-space Hamiltonian so that both RDMs and the orbital diagnostics are expressed consistently in the natural-orbital representation:

In [ ]:
# Rotate the valence orbitals using the CASCI one-particle RDM so each
# natural orbital has a well-defined correlated occupation.
natural_orbital_localizer = create("orbital_localizer", "qdk_natural_orbitals")
natural_orbital_wavefunction = natural_orbital_localizer.run(
    valence_casci_wavefunction,
    valence_indices,
    valence_indices,
)

# Rebuild and solve in the rotated basis so the RDMs and orbital entropies
# describe the same natural-orbital representation.
natural_orbital_hamiltonian = hamiltonian_constructor.run(
    natural_orbital_wavefunction.get_orbitals()
)
natural_orbital_energy, natural_orbital_casci_wavefunction = casci_solver.run(
    natural_orbital_hamiltonian,
    num_valence_alpha,
    num_valence_beta,
)
# Store ordinary Python floats rather than library scalar types so the
# values can be printed and passed to the visualization notebook directly.
orbital_entropies = [
    float(value)
    for value in natural_orbital_casci_wavefunction.get_single_orbital_entropies()
]

The two CASCI energies should agree to the displayed precision because both calculations span the same complete active space.
The second calculation is needed for the orbital-resolved selection evidence, not to lower the energy.

## Active-space refinement with orbital entropies

The single-orbital entropy used here is the [von Neumann entropy ↗](https://en.wikipedia.org/wiki/Von_Neumann_entropy) of the reduced density matrix for one spatial orbital.
Its eigenvalues are the probabilities $\omega_{a,i}$ of the four local occupation states, so the entropy has the Shannon form

$$
s_i^{(1)} = -\sum_{a=1}^{4} \omega_{a,i}\ln\omega_{a,i}.
$$

These probabilities come from diagonal elements of the spin-resolved one- and two-particle RDMs.
If $n_{i\alpha}$ and $n_{i\beta}$ are the one-particle occupations and $d_i$ is the *double-occupancy probability*—the probability that the $\alpha$ and $\beta$ spin orbitals belonging to spatial orbital $i$ are occupied simultaneously—then

$$
\begin{aligned}
\omega_{\mathrm{empty},i} &= 1-n_{i\alpha}-n_{i\beta}+d_i, \\
\omega_{\alpha,i} &= n_{i\alpha}-d_i, \\
\omega_{\beta,i} &= n_{i\beta}-d_i, \\
\omega_{\mathrm{double},i} &= d_i.
\end{aligned}
$$

The one-particle occupation $n_{i\alpha}$ includes both the $\alpha$-only and doubly occupied cases, so subtracting $d_i$ isolates the $\alpha$-only probability; the same reasoning gives the $\beta$-only probability.
The empty probability is the remainder after accounting for either spin occupation, with $d_i$ added back because double occupation was subtracted twice.
The four probabilities therefore sum to one.

An entropy near zero means that one local occupation state consistently dominates.
A larger entropy means that the orbital changes among several local occupation states across the important determinants.
In other words, the important determinants assign different occupations to this orbital together with corresponding occupation differences elsewhere in the active space.
This correlated variation makes the orbital a stronger candidate for explicit treatment.
These high-entropy orbitals carry the strongest static-correlation signal because their occupations vary among the important determinants.
Freezing a high-entropy orbital would prevent its occupation from changing with the occupations of the other orbitals and would therefore remove an important part of the multi-configurational wavefunction.
By contrast, a low-entropy orbital remains close to one local occupation state and is a better candidate to freeze as inactive or virtual.
QDK/Chemistry evaluates these probabilities and entropies from the RDMs stored in the CASCI wavefunction.
Automated active-space selection (autoCAS) uses orbital entropies to identify which orbitals should remain active.
The resulting data flow is therefore: the correlated wavefunction determines the local-state probabilities, those probabilities determine one entropy for each orbital, and autoCAS uses the entropies to select the orbitals in the active space.

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why does autoCAS require a correlated calculation before it can select orbitals?</summary>

<div style="padding:0.1em 1em;">

The selector uses single-orbital entropies derived from local occupation probabilities.
Those probabilities require one- and two-particle RDMs from a correlated wavefunction; a Hartree–Fock determinant alone does not provide the required correlation evidence.

</div>

</details></div>

The QDK/Chemistry qdk_autocas_eos selector sorts the orbital entropies and selects a high-entropy group separated from the remaining orbitals by a sufficiently large gap.
The thresholds are configurable; see [Active-space selection ↗](https://microsoft.github.io/qdk-chemistry/user/comprehensive/algorithms/active_space.html) for their defaults and use with less clearly separated entropy values.
The selector then repartitions the orbitals according to the selected group:

In [ ]:
# autoCAS uses the RDM-derived orbital entropies to retain the orbitals that
# carry the strongest correlation in a smaller active space.
autocas_selector = create("active_space_selector", "qdk_autocas_eos")
refined_wavefunction = autocas_selector.run(natural_orbital_casci_wavefunction)
refined_orbitals = refined_wavefunction.get_orbitals()

# Summarize the inactive, selected active, and virtual spatial-orbital spaces.
alpha_channel = SymmetryLabel([axes.alpha()])
refined_indices = list(refined_orbitals.active_indices().indices(alpha_channel))
if not refined_indices:
    raise RuntimeError(
        "autoCAS selected no active orbitals. Set refined_wavefunction to "
        "natural_orbital_wavefunction to retain the complete valence space, "
        "or adjust the autoCAS thresholds before continuing."
    )
inactive_indices = list(refined_orbitals.inactive_indices().indices(alpha_channel))
num_refined_alpha, num_refined_beta = (
    refined_wavefunction.get_active_num_electrons()
)
num_refined_electrons = num_refined_alpha + num_refined_beta
num_refined_orbitals = len(refined_indices)
num_virtual_orbitals = (
    refined_orbitals.get_num_molecular_orbitals()
    - len(inactive_indices)
    - num_refined_orbitals
)

# Natural occupations can be degenerate, leaving the corresponding orbital
# vectors free to rotate. Choose that gauge only after autoCAS has selected
# the final active subspace, because lambda belongs to its mapped Hamiltonian.
coordinate_minimization = coordinate_minimize_natural_orbital_coefficient_norm(
    valence_casci_wavefunction,
    refined_orbitals,
    valence_indices,
)
refined_orbitals = coordinate_minimization.orbitals

<div style="text-align:center;">

<svg xmlns:xlink="http://www.w3.org/1999/xlink" width="526.79952pt" height="310.794102pt" viewBox="0 0 526.79952 310.794102" xmlns="http://www.w3.org/2000/svg" version="1.1" class="qdk-chemistry-orbital-entropy" role="img" data-asset="tutorial_qpe_orbital_entropy.svg" aria-label="Entropy-ranked candidate orbitals. Selected orbitals 8, 7, 5, 6, 9, and 4 have entropies of approximately 0.966, 0.966, 0.964, 0.964, 0.554, and 0.548. Excluded orbitals 3 and 2 have entropies of approximately 0.030 and 0.022. A dashed vertical cut separates the sixth and seventh entropy ranks." style="max-width:100%;height:auto"> <metadata> <rdf:RDF xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:cc="http://creativecommons.org/ns#" xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#"> <cc:Work> <dc:type rdf:resource="http://purl.org/dc/dcmitype/StillImage"/> <dc:format>image/svg+xml</dc:format> <dc:creator> <cc:Agent> <dc:title>Matplotlib v3.11.1, https://matplotlib.org/</dc:title> </cc:Agent> </dc:creator> </cc:Work> </rdf:RDF> </metadata> <defs> <style type="text/css"> .qdk-chemistry-orbital-entropy { --qdk-host-foreground: var( --vscode-editor-foreground, var(--jp-widgets-color, currentColor) ); --diagram-foreground: var(--qdk-host-foreground); --diagram-grid: var( --vscode-panel-border, var(--jp-border-color2, #888888) ); } .qdk-chemistry-orbital-entropy text { fill: var(--diagram-foreground) !important; } </style>  <style type="text/css">.qdk-chemistry-orbital-entropy *{stroke-linejoin: round; stroke-linecap: butt}</style> </defs> <g id="figure_1"> <g id="patch_1"> <path d="M 0 310.794102 L 526.79952 310.794102 L 526.79952 0 L 0 0 L 0 310.794102 z " style="fill: none"/> </g> <g id="axes_1"> <g id="patch_2"> <path d="M 44.103906 272.593321 L 519.59952 272.593321 L 519.59952 7.538634 L 44.103906 7.538634 L 44.103906 272.593321 z " style="fill: none"/> </g> <g id="line2d_1"> <path d="M 65.717343 19.890027 L 127.47002 19.890027 L 189.222697 20.446435 L 250.975375 20.446435 L 312.728052 127.667527 L 374.480729 129.277299 L 436.233406 264.755207 L 497.986083 266.917882 " clip-path="url(#p7b563b744f)" style="fill: none; stroke: #455a64; stroke-width: 1.5; stroke-linecap: square"/> </g> <g id="matplotlib.axis_1"> <g id="xtick_1"> <g id="line2d_2"> <defs> <path id="m47589d7333" d="M 0 0 L 0 3.5 " style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </defs> <g> <use xlink:href="#m47589d7333" x="65.717343" y="272.593321" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_1"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: middle" x="65.717343" y="287.190977" transform="rotate(-0 65.717343 287.190977)">7</text> </g> </g> <g id="xtick_2"> <g id="line2d_3"> <g> <use xlink:href="#m47589d7333" x="127.47002" y="272.593321" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_2"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: middle" x="127.47002" y="287.190977" transform="rotate(-0 127.47002 287.190977)">8</text> </g> </g> <g id="xtick_3"> <g id="line2d_4"> <g> <use xlink:href="#m47589d7333" x="189.222697" y="272.593321" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_3"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: middle" x="189.222697" y="287.190977" transform="rotate(-0 189.222697 287.190977)">6</text> </g> </g> <g id="xtick_4"> <g id="line2d_5"> <g> <use xlink:href="#m47589d7333" x="250.975375" y="272.593321" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_4"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: middle" x="250.975375" y="287.190977" transform="rotate(-0 250.975375 287.190977)">5</text> </g> </g> <g id="xtick_5"> <g id="line2d_6"> <g> <use xlink:href="#m47589d7333" x="312.728052" y="272.593321" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_5"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: middle" x="312.728052" y="287.190977" transform="rotate(-0 312.728052 287.190977)">9</text> </g> </g> <g id="xtick_6"> <g id="line2d_7"> <g> <use xlink:href="#m47589d7333" x="374.480729" y="272.593321" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_6"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: middle" x="374.480729" y="287.190977" transform="rotate(-0 374.480729 287.190977)">4</text> </g> </g> <g id="xtick_7"> <g id="line2d_8"> <g> <use xlink:href="#m47589d7333" x="436.233406" y="272.593321" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_7"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: middle" x="436.233406" y="287.190977" transform="rotate(-0 436.233406 287.190977)">3</text> </g> </g> <g id="xtick_8"> <g id="line2d_9"> <g> <use xlink:href="#m47589d7333" x="497.986083" y="272.593321" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_8"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: middle" x="497.986083" y="287.190977" transform="rotate(-0 497.986083 287.190977)">2</text> </g> </g> <g id="text_9"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: middle" x="281.851713" y="301.191758" transform="rotate(-0 281.851713 301.191758)">Natural-orbital index (sorted by decreasing entropy)</text> </g> </g> <g id="matplotlib.axis_2"> <g id="ytick_1"> <g id="line2d_10"> <defs> <path id="m5057f78d95" d="M 0 0 L -3.5 0 " style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </defs> <g> <use xlink:href="#m5057f78d95" x="44.103906" y="272.593321" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_10"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: end" x="37.103906" y="276.392149" transform="rotate(-0 37.103906 276.392149)">0.0</text> </g> </g> <g id="ytick_2"> <g id="line2d_11"> <g> <use xlink:href="#m5057f78d95" x="44.103906" y="220.274422" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_11"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: end" x="37.103906" y="224.07325" transform="rotate(-0 37.103906 224.07325)">0.2</text> </g> </g> <g id="ytick_3"> <g id="line2d_12"> <g> <use xlink:href="#m5057f78d95" x="44.103906" y="167.955524" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_12"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: end" x="37.103906" y="171.754352" transform="rotate(-0 37.103906 171.754352)">0.4</text> </g> </g> <g id="ytick_4"> <g id="line2d_13"> <g> <use xlink:href="#m5057f78d95" x="44.103906" y="115.636625" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_13"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: end" x="37.103906" y="119.435453" transform="rotate(-0 37.103906 119.435453)">0.6</text> </g> </g> <g id="ytick_5"> <g id="line2d_14"> <g> <use xlink:href="#m5057f78d95" x="44.103906" y="63.317727" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_14"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: end" x="37.103906" y="67.116555" transform="rotate(-0 37.103906 67.116555)">0.8</text> </g> </g> <g id="ytick_6"> <g id="line2d_15"> <g> <use xlink:href="#m5057f78d95" x="44.103906" y="10.998828" style="stroke: var(--diagram-foreground); stroke-width: 0.8"/> </g> </g> <g id="text_15"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: end" x="37.103906" y="14.797656" transform="rotate(-0 37.103906 14.797656)">1.0</text> </g> </g> <g id="text_16"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: middle" x="14.798438" y="140.065978" transform="rotate(-90 14.798438 140.065978)">Single-orbital entropy</text> </g> </g> <g id="PathCollection_1"> <defs> <path id="ma309a65d5c" d="M 0 3.708099 C 0.983399 3.708099 1.926654 3.317391 2.622022 2.622022 C 3.317391 1.926654 3.708099 0.983399 3.708099 0 C 3.708099 -0.983399 3.317391 -1.926654 2.622022 -2.622022 C 1.926654 -3.317391 0.983399 -3.708099 0 -3.708099 C -0.983399 -3.708099 -1.926654 -3.317391 -2.622022 -2.622022 C -3.317391 -1.926654 -3.708099 -0.983399 -3.708099 0 C -3.708099 0.983399 -3.317391 1.926654 -2.622022 2.622022 C -1.926654 3.317391 -0.983399 3.708099 0 3.708099 z " style="stroke: #00796b"/> </defs> <g clip-path="url(#p7b563b744f)"> <use xlink:href="#ma309a65d5c" x="65.717343" y="19.890027" style="fill: #00796b; stroke: #00796b"/> <use xlink:href="#ma309a65d5c" x="127.47002" y="19.890027" style="fill: #00796b; stroke: #00796b"/> <use xlink:href="#ma309a65d5c" x="189.222697" y="20.446435" style="fill: #00796b; stroke: #00796b"/> <use xlink:href="#ma309a65d5c" x="250.975375" y="20.446435" style="fill: #00796b; stroke: #00796b"/> <use xlink:href="#ma309a65d5c" x="312.728052" y="127.667527" style="fill: #00796b; stroke: #00796b"/> <use xlink:href="#ma309a65d5c" x="374.480729" y="129.277299" style="fill: #00796b; stroke: #00796b"/> </g> </g> <g id="PathCollection_2"> <defs> <path id="mff9feb2639" d="M -3.708099 3.708099 L 3.708099 3.708099 L 3.708099 -3.708099 L -3.708099 -3.708099 z " style="stroke: #7b1fa2"/> </defs> <g clip-path="url(#p7b563b744f)"> <use xlink:href="#mff9feb2639" x="436.233406" y="264.755207" style="fill: #7b1fa2; stroke: #7b1fa2"/> <use xlink:href="#mff9feb2639" x="497.986083" y="266.917882" style="fill: #7b1fa2; stroke: #7b1fa2"/> </g> </g> <g id="line2d_16"> <path d="M 405.357067 272.593321 L 405.357067 7.538634 " clip-path="url(#p7b563b744f)" style="fill: none; stroke-dasharray: 5.55,2.4; stroke-dashoffset: 0; stroke: #5c6bc0; stroke-width: 1.5"/> </g> <g id="patch_3"> <path d="M 44.103906 272.593321 L 44.103906 7.538634 " style="fill: none; stroke: var(--diagram-foreground); stroke-width: 0.8; stroke-linejoin: miter; stroke-linecap: square"/> </g> <g id="patch_4"> <path d="M 44.103906 272.593321 L 519.59952 272.593321 " style="fill: none; stroke: var(--diagram-foreground); stroke-width: 0.8; stroke-linejoin: miter; stroke-linecap: square"/> </g> <g id="legend_1"> <g id="PathCollection_3"> <g> <use xlink:href="#ma309a65d5c" x="63.103906" y="228.564414" style="fill: #00796b; stroke: #00796b"/> </g> </g> <g id="text_17"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: start" x="81.103906" y="231.189414" transform="rotate(-0 81.103906 231.189414)">Selected by autoCAS</text> </g> <g id="PathCollection_4"> <g> <use xlink:href="#mff9feb2639" x="63.103906" y="243.565196" style="fill: #7b1fa2; stroke: #7b1fa2"/> </g> </g> <g id="text_18"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: start" x="81.103906" y="246.190196" transform="rotate(-0 81.103906 246.190196)">Excluded</text> </g> <g id="line2d_17"> <path d="M 53.103906 257.690977 L 63.103906 257.690977 L 73.103906 257.690977 " style="fill: none; stroke-dasharray: 5.55,2.4; stroke-dashoffset: 0; stroke: #5c6bc0; stroke-width: 1.5"/> </g> <g id="text_19"> <text style="font-size: 10px; font-family: 'DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'Lucida Grande', 'Verdana', 'Geneva', 'Lucid', 'Arial', 'Helvetica', 'Avant Garde', sans-serif; text-anchor: start" x="81.103906" y="261.190977" transform="rotate(-0 81.103906 261.190977)">autoCAS cut</text> </g> </g> </g> </g> <defs> <clipPath id="p7b563b744f"> <rect x="44.103906" y="7.538634" width="475.495614" height="265.054686"/> </clipPath> </defs> </svg>

*Candidate natural orbitals sorted by decreasing single-orbital entropy. autoCAS retains the high-entropy group to the left of the dashed cut.*

</div>

The asterisks in the script output identify the selected orbitals.
The selected high-entropy group determines the refined active space.
Equal natural-orbital occupations can leave the corresponding orbital vectors free to rotate within a degenerate subspace.
The script chooses a reproducible representation within each selected degenerate block by coordinate-minimizing the mapped Hamiltonian coefficient norm $\lambda=\sum_\ell\lvert h_\ell\rvert$, without changing the orbital subspace or its exact CASCI energy.
Among the unselected orbitals, those below the occupied–virtual boundary of the reference determinant become inactive, while those above the boundary become virtual.
Freezing these low-entropy orbitals is still an approximation because low entropy does not mean that their correlation contribution is exactly zero, so the energy comparison below measures part of its cost.
Their entropies are small rather than exactly zero, and allowing excitations involving them can still lower the correlated energy.

## The algorithmic reference

The script finishes by solving the refined active-space Hamiltonian with CASCI:

In [ ]:
refined_hamiltonian = hamiltonian_constructor.run(refined_orbitals)
refined_energy, refined_casci_wavefunction = casci_solver.run(
    refined_hamiltonian,
    num_refined_alpha,
    num_refined_beta,
)
num_refined_determinants = len(refined_casci_wavefunction.get_coefficients())

The resulting determinant count quantifies the reduction in problem size for the quantum-computing stages of the tutorial.
The final CASCI energy is the exact ground-state energy of the selected active-space Hamiltonian, up to numerical solver tolerance, and will be the *algorithmic reference energy* for state preparation and phase estimation.
CASCI is a full configuration-interaction calculation within the selected active space, but it is not the exact energy of N<sub>2</sub> in the full `cc-pvdz` orbital space: fixing the inactive orbitals as doubly occupied and the virtual orbitals as empty excludes correlation involving those orbitals.

## The active-space choice

The initial valence space includes more orbitals than the refined active space so that the correlated calculation can first measure the entropy of every candidate orbital.
The refinement then uses this evidence to decide which orbital occupations must remain variable and which can be frozen.
Freezing additional orbital occupations cannot lower the CASCI energy.
It leaves the energy unchanged only if the removed determinants contribute nothing to the larger-space ground state; otherwise, as in this example, the energy increases.
The script reports the observed energy increase when reducing the active space.

The observed increase quantifies correlation excluded when reducing the initial valence space.
This active-space model error is separate from the $1\ \mathrm{m}E_{\mathrm{h}}$ teaching target, which applies only to the later quantum algorithm's agreement with the compact-model CASCI reference.

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why should the energy increase caused by active-space refinement not be judged against the 1&nbsp;m<i>E</i><sub>h</sub> teaching target?</summary>

<div style="padding:0.1em 1em;">

The energy increase measures correlation excluded when orbital occupations are frozen during active-space refinement.
The $1\ \mathrm{m}E_{\mathrm{h}}$ target applies later when comparing the phase-estimation energy with the exact CASCI energy of the same selected-space Hamiltonian.
These comparisons measure different approximations.

</div>

</details></div>

For this tutorial, the refined active space is accepted as a compact model because it retains the orbitals with the strongest entropy-based correlation evidence while producing a tractable Hamiltonian for validating the quantum workflow.
The energy difference from the initial valence-space calculation remains documented as model error.
The next chapter will determine how the selected active spatial orbitals are represented by qubits.

## Count the determinants

The refined active space is CAS $(6,6)$: six electrons in six spatial orbitals, so three $\alpha$ and three $\beta$ electrons. The $\alpha$ and $\beta$ occupations are chosen independently of each other.

Fix the function below so that it returns the number of determinants this active space contains, then run the cell.

In [ ]:
from math import comb

from _unit import exercise


@exercise
def determinant_count():
    alpha = comb(6, 3)
    beta = comb(6, 3)
    return alpha

**Hint**

`alpha` and `beta` already count the arrangements for each spin on its own. Nothing ties the two choices together, so ask yourself how many whole-determinant arrangements a single $\alpha$ choice can appear in.

In [ ]:
@exercise
def determinant_count():
    alpha = comb(6, 3)
    beta = comb(6, 3)
    return alpha * beta

The function returns the product of the two independent occupation counts. $\binom{6}{3}=20$, so the refined space holds $20\times20=400$ determinants, down from the 3,136 of the initial CAS $(10,8)$ valence space.

## Running the calculation

The cells above have already run the workflow. Use their output to answer the questions below.

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;What initial valence space and determinant count did the script construct?</summary>

<div style="padding:0.1em 1em;">

The script reports ten active electrons in eight active spatial orbitals, written CAS $(10,8)$, with five $\alpha$ and five $\beta$ active electrons.
The active orbital indices are 2 through 9.
Of the 28 `cc-pvdz` molecular orbitals, indices 0 and 1 are initially inactive and the remaining 18 are virtual.
The determinant count is $\binom{8}{5}\binom{8}{5}=3136$.

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Which orbitals did autoCAS retain, and how much did refinement reduce the problem size?</summary>

<div style="padding:0.1em 1em;">

Orbitals 4 through 9 have entropies from approximately 0.548 to 0.966, separated by a large gap from the remaining values of approximately 0.030 or less.
autoCAS retains these six orbitals in CAS $(6,6)$, containing three $\alpha$ and three $\beta$ active electrons.
The refined partition has four inactive orbitals, six active orbitals, and 18 virtual orbitals.
Its determinant count is $\binom{6}{3}\binom{6}{3}=400$, compared with 3,136 determinants in the initial valence space.

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Did the natural-orbital transformation change the CASCI energy?</summary>

<div style="padding:0.1em 1em;">

No change appears at the displayed precision.
The script reports the signed energy change after the transformation, which is consistent with numerical roundoff near zero.
Both calculations span the same complete active subspace, so changing the orbital representation does not change the exact CASCI energy within that subspace.

</div>

</details></div>

Use the final selected-space energy as the algorithmic reference, while retaining the larger-space result as evidence of the correlation excluded by the compact model.

## Candidate-orbital visualization

The viewer below displays every candidate natural orbital from the initial valence space, including orbitals that autoCAS did not retain.
Use the viewer to inspect the following information:

- **Orbital menu**<br>
  Selects each candidate natural orbital for comparison.
  The menu follows increasing molecular-orbital index, which corresponds here to decreasing natural occupation.
  The menu is not ordered by entropy.
- **Isosurface**<br>
  Traces points where the orbital wavefunction has a chosen positive or negative value, revealing its lobes, nodes, and spatial extent.
  The surface itself does not encode occupation or entropy.
- **Natural occupation**<br>
  Reports the average number of electrons in the spatial orbital.
  A value near two indicates an almost always doubly occupied orbital, a value near zero indicates an almost always empty orbital, and an intermediate value indicates variable occupation across the correlated wavefunction.
- **Single-orbital entropy and autoCAS selection**<br>
  Reports the uncertainty in the orbital's local occupation and whether autoCAS retained it.
  Larger entropy indicates stronger coupling to the occupations of the other active orbitals.

autoCAS selects the strongly coupled group from gaps in the orbital entropies, not from orbital shapes or a cutoff applied to the natural occupations.
Use the shapes as aids to chemical interpretation, but defend the final active space using the numerical occupation and entropy evidence in the overlays.

The molecular viewer needs values of each orbital's spatial wavefunction on a three-dimensional grid. The next cell evaluates all candidate natural orbitals on such a grid and stores the sampled values as cube data.

Each orbital is annotated with its natural occupation, single-orbital entropy, and autoCAS selection status so that you can examine the spatial and numerical evidence together. Generating the orbital grids may take a little longer than the preceding calculation.

In [ ]:
from qdk_chemistry.utils.cubegen import generate_cubefiles_from_orbitals

natural_orbitals = natural_orbital_casci_wavefunction.get_orbitals()
occupation_alpha, occupation_beta = (
    natural_orbital_casci_wavefunction.get_active_orbital_occupations()
)

# Occupation arrays use active-space positions, while cube-file labels use
# the original molecular-orbital indices; this dictionary connects them.
active_position = {
    orbital_index: position for position, orbital_index in enumerate(valence_indices)
}
raw_cube_data = generate_cubefiles_from_orbitals(
    orbitals=natural_orbitals,
    grid_size=(30, 30, 30),
    margin=10.0,
    indices=valence_indices,
)

cube_data = {}
for raw_label, cube_file in raw_cube_data.items():
    # Cube labels number orbitals from one, while QDK/Chemistry indices start
    # from zero, so convert before looking up occupations and entropies.
    orbital_index = int(raw_label.split("_")[1]) - 1
    position = active_position[orbital_index]
    # Add the alpha and beta occupations to report the total occupation of
    # each spatial orbital in the viewer.
    occupation = float(occupation_alpha[position]) + float(occupation_beta[position])
    cube_data[f"Orbital {orbital_index}"] = {
        "data": cube_file,
        "info": {
            "Occupation": f"{occupation:.3f}",
            "Entropy": f"{orbital_entropies[position]:.3f}",
            "Selected by autoCAS": "yes" if orbital_index in refined_indices else "no",
        },
    }

print(f"Generated cube data for {len(cube_data)} candidate orbitals.")

## Inspect the natural orbitals

Launch the interactive molecular viewer and use its orbital menu to move through the candidate natural orbitals. For each orbital, inspect the isosurface together with the displayed "occupation", "entropy", and "selected by autoCAS" information. Compare the selected orbitals with the nearly doubly occupied and nearly empty orbitals that were excluded.

In [ ]:
from qdk.widgets import MoleculeViewer

MoleculeViewer(
    molecule_data=structure.to_xyz(),
    cube_data=cube_data,
)

Use the viewer to inspect the following information:

- **Orbital menu** selects each candidate natural orbital for comparison. The menu follows increasing molecular-orbital index, which corresponds here to decreasing natural occupation. The menu is not ordered by entropy.
- **Isosurface** traces points where the orbital wavefunction has a chosen positive or negative value, revealing its lobes, nodes, and spatial extent. The surface itself does not encode occupation or entropy.
- **Natural occupation** reports the average number of electrons in the spatial orbital. A value near two indicates an almost always doubly occupied orbital, a value near zero indicates an almost always empty orbital, and an intermediate value indicates variable occupation across the correlated wavefunction.
- **Single-orbital entropy and autoCAS selection** report the uncertainty in the orbital's local occupation and whether autoCAS retained it. Larger entropy indicates stronger coupling to the occupations of the other active orbitals.

autoCAS selects the strongly coupled group from gaps in the orbital entropies, not from orbital shapes or a cutoff applied to the natural occupations. Use the shapes as aids to chemical interpretation, but defend the final active space using the numerical occupation and entropy evidence in the overlays.

## Interpret the active-space choice

Identify which visual features distinguish nearly doubly occupied orbitals, high-entropy orbitals with strongly coupled occupations, and nearly empty orbitals. Then explain why autoCAS retains the selected group while excluding the other candidate orbitals.

Treat the orbital shapes as aids to chemical interpretation rather than as the selection rule. Defend the refined active-space choice using the occupation and entropy overlays.

## Further reading

- [Orbital localization and transformation ↗](https://microsoft.github.io/qdk-chemistry/user/comprehensive/algorithms/localizer.html)
- [Active-space selection ↗](https://microsoft.github.io/qdk-chemistry/user/comprehensive/algorithms/active_space.html)
- [Multi-configuration calculations ↗](https://microsoft.github.io/qdk-chemistry/user/comprehensive/algorithms/mc_calculator.html)
- [Wavefunctions ↗](https://microsoft.github.io/qdk-chemistry/user/comprehensive/data/wavefunction.html)

<div style="display:flex;justify-content:space-between;align-items:stretch;gap:12px;margin-top:8px"><a href="../02-describe-molecule/describe_molecule.workbook.ipynb" title="Previous unit: Describing the Molecule" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-secondaryBackground, transparent);color:var(--vscode-button-secondaryForeground, inherit);border-color:var(--vscode-widget-border, #8884)"><span aria-hidden="true">&#8249;</span><span><span style="display:block;font-size:11px;opacity:.75">Previous unit</span><span style="display:block;font-weight:600">Describing the Molecule</span></span></a><a href="../04-map-to-qubits/map_to_qubits.workbook.ipynb" title="Next unit: Mapping the Problem to Qubits" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-background, #005fb8);color:var(--vscode-button-foreground, #ffffff);border-color:var(--vscode-button-background, #005fb8)"><span style="text-align:right"><span style="display:block;font-size:11px;opacity:.75">Next unit</span><span style="display:block;font-weight:600">Mapping the Problem to Qubits</span></span><span aria-hidden="true">&#8250;</span></a></div>